# DBTS Test Analysis

This notebook loads the DBTS outputs (generated by `strategy/dbts_runner.py`) and produces a research-style analysis for the TEST period. Run top-to-bottom.

In [ ]:
# Imports and plotting setup
import os
import math
import warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

sns.set(style="whitegrid", context="talk")
%matplotlib inline
warnings.filterwarnings("ignore")

In [ ]:
# Paths to DBTS outputs (written by strategy/dbts_runner.py)
OUT_DIR = 'outputs'
TRADE_LOG = os.path.join(OUT_DIR, 'dbts_test_trade_log.csv')
DAILY_SCORES = os.path.join(OUT_DIR, 'dbts_daily_target_scores.csv')
ORIG_BT_METRICS = os.path.join(OUT_DIR, 'backtest_metrics.json')  # optional

print('Reading files:', TRADE_LOG, DAILY_SCORES)
assert os.path.exists(TRADE_LOG), f'Missing {TRADE_LOG} - run dbts_runner first'
assert os.path.exists(DAILY_SCORES), f'Missing {DAILY_SCORES} - run dbts_runner first'

trades = pd.read_csv(TRADE_LOG, parse_dates=['date','entry_date','exit_date'])
daily_scores = pd.read_csv(DAILY_SCORES, parse_dates=['date'])

print('Loaded', len(trades), 'trade rows and', len(daily_scores), 'daily-score rows')

## Section 1 — Run Information

In [ ]:
# Attempt to load market data and compute chrono split (best-effort).
from strategy.strategy_config import StrategyConfig
from data.loader import Loader
from strategy.splits import chrono_split

cfg = StrategyConfig()
print('StrategyConfig start:', cfg.start)
# Load market data (may use cache)
loader = Loader()
md = loader.load()
split = chrono_split(md.prices.index, cfg)
train_span = (split.train_idx[0].date(), split.train_idx[-1].date()) if len(split.train_idx) else (None,None)
val_span = (split.val_idx[0].date(), split.val_idx[-1].date()) if len(split.val_idx) else (None,None)
test_span = (split.test_idx[0].date(), split.test_idx[-1].date()) if len(split.test_idx) else (None,None)
n_sectors = len([k for k in cfg.to_dict().get('cache_dir', {})]) if False else len(md.prices.columns)  # placeholder
# Better: derive from config SECTORS
from config import SECTORS
n_sectors = len(SECTORS)
n_stocks = len(md.prices.columns)
n_candidates = max( (1 if 'target' in v else 0) + len(v.get('predictors', [])) for v in SECTORS.values() )
n_days = len(md.prices.index)

info = dict(train_period=train_span, val_period=val_span, test_period=test_span,
            n_sectors=n_sectors, n_stocks=n_stocks, n_candidates=n_candidates, n_days=n_days)

import pprint
pprint.pprint(info)

## Section 2 — Overall Performance (TEST period)

In [ ]:
# Compute performance metrics from trade-level realized returns
# Use exit_date as the P&L realization day.
tr = trades.copy()
tr = tr.dropna(subset=['realized_return']).reset_index(drop=True)
# cumulative return as product(1+r)-1
cum_ret = (1 + tr['realized_return']).prod() - 1 if len(tr) else 0.0
# build daily returns series from realized P&L posted on exit_date
daily = tr.groupby('exit_date')['realized_return'].sum().reindex(pd.date_range(start=split.test_idx[0], end=split.test_idx[-1], freq='B'), fill_value=0.0)
daily.index.name = 'date'
ndays = len(daily)
# annualized return (CAGR-like)
ann_ret = (1 + daily.cumsum().iloc[-1]) ** (252.0 / ndays) - 1 if ndays>0 else 0.0
# daily return series for vol
daily_ret = daily.values
ann_vol = float(np.nanstd(daily_ret, ddof=1) * np.sqrt(252)) if ndays>1 else float('nan')
sharpe = ann_ret / ann_vol if ann_vol and ann_vol>0 else float('nan')
# Sortino: downside std
downside = daily_ret[daily_ret<0] if len(daily_ret) else np.array([])
down_std = np.std(downside, ddof=1) * np.sqrt(252) if len(downside)>1 else 0.0
sortino = ann_ret / down_std if down_std>0 else float('nan')
# equity curve and drawdown
equity = (1 + daily).cumprod()
peak = equity.cummax()
drawdown = equity / peak - 1
max_dd = float(drawdown.min())
calmar = ann_ret / abs(max_dd) if max_dd<0 else float('nan')
# trade stats
n_trades = len(tr)
win_rate = float((tr['realized_return']>0).mean()) if n_trades else float('nan')
avg_trade = float(tr['realized_return'].mean()) if n_trades else float('nan')
med_trade = float(tr['realized_return'].median()) if n_trades else float('nan')
n_long = int((tr['signal']==1).sum()) if 'signal' in tr else 0
n_short = int((tr['signal']==-1).sum()) if 'signal' in tr else 0

metrics = dict(cumulative_return=cum_ret, annualized_return=ann_ret, annualized_vol=ann_vol,
               sharpe=sharpe, sortino=sortino, max_drawdown=max_dd, calmar=calmar,
               win_rate=win_rate, n_trades=n_trades, avg_trade_return=avg_trade, median_trade_return=med_trade,
               n_long=n_long, n_short=n_short)
import pprint
pprint.pprint(metrics)

In [ ]:
# Pretty table and simple plots
display(pd.DataFrame([metrics]).T.rename(columns={0:'value'}))
fig, ax = plt.subplots(2,2, figsize=(14,10))
equity.plot(ax=ax[0,0], title='Strategy equity (DBTS)')
ax[0,0].set_ylabel('Equity')
drawdown.plot(ax=ax[0,1], title='Drawdown')
ax[0,1].set_ylabel('Drawdown')
daily.rolling(20).std().mul(np.sqrt(252)).plot(ax=ax[1,0], title='Rolling vol (20d)')
(daily.rolling(60).mean()*np.sqrt(252)).plot(ax=ax[1,1], title='Rolling mean daily ret (60)')
plt.tight_layout()
plt.show()

## Section 3 — Equity Curve Analysis (vs buy-and-hold)

In [ ]:
# Buy-and-hold for an equally-weighted basket of sector targets over test period
from config import ALL_TARGETS
test_dates = pd.DatetimeIndex(split.test_idx)
prices_test = md.prices[ALL_TARGETS].reindex(test_dates)
prices_test = prices_test.dropna(axis=1, how='all')
# normalize to 1 at start
bnh = prices_test / prices_test.iloc[0]
bnh_eq = bnh.mean(axis=1)
fig, ax = plt.subplots(1,1, figsize=(12,5))
equity.plot(ax=ax, label='DBTS equity')
bnh_eq.cumprod().plot(ax=ax, label='Buy-and-hold (targets eq-weight)')
ax.legend()
ax.set_title('Equity: DBTS vs Buy-and-hold')
plt.show()

## Section 4 — Target Selection Analysis

In [ ]:
# Aggregations per sector / candidate from daily_scores
ds = daily_scores.copy()
# ensure sector column exists (daily_scores saved sector name)
grp = ds.groupby(['sector','candidate'])
agg = grp.agg(count=('final_score','count'), avg_final_score=('final_score','mean'),
               avg_bandit=('bandit_score','mean'), avg_residual=('residual_score','mean'), avg_adf=('adf_score','mean'))
agg = agg.reset_index().sort_values(['sector','count'], ascending=[True,False])
display(agg.head(40))
# bar chart for a sample sector
sample_sector = agg['sector'].unique()[0] if len(agg) else None
if sample_sector is not None:
    s = agg[agg['sector']==sample_sector].set_index('candidate')
    s[['count','avg_bandit','avg_final_score']].plot(kind='bar', subplots=True, layout=(1,3), figsize=(16,4), legend=False)
    plt.suptitle(f'Target selection summary — {sample_sector}')
    plt.show()

## Section 5 — Bandit Learning Diagnostics

In [ ]:
# Reconstruct alpha/beta evolution from trade log updates (ordered by exit_date)
updates = trades.dropna(subset=['exit_date']).sort_values('exit_date')
# For each sector+target, track alpha/beta starting from 1/1
from collections import defaultdict
state_series = defaultdict(list)  # (sector,target) -> list of (date,alpha,beta)
alpha_beta = {}
# initialize entries for all candidates from daily_scores
for _, row in daily_scores.groupby('sector'):
    pass
# instead initialize from unique (sector,candidate) pairs
for sector, cand in daily_scores[['sector','candidate']].drop_duplicates().values:
    alpha_beta[(sector,cand)] = [1,1]
# apply updates in chronological order using alpha_before/after in trades where available
for _, r in updates.iterrows():
    sector = r['sector']
    target = r['selected_target']
    date = r['exit_date'] if pd.notna(r['exit_date']) else r['date']
    a_before = int(r['alpha_before']) if not pd.isna(r.get('alpha_before')) else alpha_beta.get((sector,target),[1,1])[0]
    b_before = int(r['beta_before']) if not pd.isna(r.get('beta_before')) else alpha_beta.get((sector,target),[1,1])[1]
    a_after = int(r['alpha_after']) if not pd.isna(r.get('alpha_after')) else a_before
    b_after = int(r['beta_after']) if not pd.isna(r.get('beta_after')) else b_before
    alpha_beta[(sector,target)] = [a_after,b_after]
    state_series[(sector,target)].append({'date': pd.to_datetime(date), 'alpha': a_after, 'beta': b_after})
# Pick a sample sector and plot top targets alpha/beta over time
sample_sector = daily_scores['sector'].unique()[0] if len(daily_scores) else None
if sample_sector is not None:
    keys = [k for k in state_series.keys() if k[0]==sample_sector]
    plt.figure(figsize=(12,6))
    for k in keys[:6]:
        dfk = pd.DataFrame(state_series[k]).set_index('date').sort_index()
        if dfk.empty:
            continue
        plt.plot(dfk.index, dfk['alpha'], label=f'{k[1]} alpha')
    plt.title(f'Alpha evolution (sample sector={sample_sector})')
    plt.legend()
    plt.show()

## Section 6 — Trade Diagnostics

In [ ]:
# Show trade table and simple charts
cols = ['date','sector','selected_target','signal','entry_date','exit_date','entry_price','exit_price','realized_return']
display(tr[cols].head(50))
# return distribution
plt.figure(figsize=(8,4))
sns.histplot(tr['realized_return'].dropna(), bins=50, kde=True)
plt.title('Trade return distribution')
plt.show()
# trade duration distribution
if 'holding_period' in tr.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(tr['holding_period'].dropna(), bins=30)
    plt.title('Trade holding period distribution')
    plt.show()
# cumulative PnL by sector
by_sector = tr.groupby('sector')['realized_return'].sum().sort_values(ascending=False)
display(by_sector)
plt.figure(figsize=(10,5))
by_sector.plot(kind='bar')
plt.title('Cumulative realized return by sector')
plt.show()

## Section 7 — Signal Diagnostics

In [ ]:
# Signals produced and executed
if 'signal' in trades.columns:
    print('Predicted signals distribution (test):')
    print(trades['signal'].value_counts(dropna=False))
# executed longs/shorts
exec_long = ((trades['signal']==1) & (trades['realized_return'].notna())).sum()
exec_short = ((trades['signal']==-1) & (trades['realized_return'].notna())).sum()
print('Executed longs:', exec_long, 'Executed shorts:', exec_short)
# skipped trades: rows where exit_price is NaN or realized_return is NaN
skipped = trades[trades['realized_return'].isna()]
print('Skipped trades count:', len(skipped))
if len(skipped):
    display(skipped.head())

## Section 8 — Classification Diagnostics

In [ ]:
# If ground-truth labels available, compute confusion matrix and report.
# The DBTS runner may not save true labels — attempt to load if present.
if 'true_label' in trades.columns:
    y_true = trades['true_label'].dropna().astype(int)
    y_pred = trades.loc[y_true.index,'signal'].astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[-1,0,1])
    print('Confusion matrix')
    display(pd.DataFrame(cm, index=['true_short','true_flat','true_long'], columns=['pred_short','pred_flat','pred_long']))
    print('
Classification report:')
    print(classification_report(y_true, y_pred, labels=[-1,0,1]))
else:
    print('True labels not available in trade log — skipping classification diagnostics')

## Section 9 — Leakage & Sanity Checks

In [ ]:
checks = {}
# 1) Bandit updates only after realized outcome: check that alpha_before/after timestamps align with exit_date ordering
if 'exit_date' in trades.columns and 'alpha_before' in trades.columns:
    bad = trades[trades['exit_date'].isna() & (trades['alpha_after'].notna())]
    checks['bandit_update_on_exit'] = 'PASS' if len(bad)==0 else 'FAIL'
else:
    checks['bandit_update_on_exit'] = 'UNKNOWN'
# 2) No NaN leakage in key columns
nan_issues = []
for c in ['entry_date','exit_date','entry_price','exit_price','realized_return']:
    if c in trades.columns and trades[c].isna().any():
        nan_issues.append(c)
checks['nan_in_trade_key_cols'] = 'PASS' if not nan_issues else f'FAIL ({nan_issues})'
# 3) Selected target was in sector universe
from config import SECTORS
valid = True
for _, r in trades.iterrows():
    s = r['sector']
    t = r['selected_target']
    # find sector key by name
    matches = [k for k,v in SECTORS.items() if v['name']==s]
    if not matches or t not in ([SECTORS[m]['target']] + SECTORS[m]['predictors'] for m in matches):
        valid = False
        break
checks['selected_in_universe'] = 'PASS' if valid else 'FAIL'
# 4) predictors excluded the selected target — unable to fully verify (predictors not logged)
checks['predictors_exclude_target'] = 'UNKNOWN (predictors not saved)'
# 5) No test information used during training — best-effort: verify dates used for training end before test start
try:
    train_end = split.train_idx[-1]
    test_start = split.val_idx[0] if len(split.val_idx) else split.test_idx[0]
    checks['train_before_test'] = 'PASS' if train_end < test_start else 'FAIL'
except Exception:
    checks['train_before_test'] = 'UNKNOWN'

display(pd.Series(checks, name='result'))

## Section 10 — DBTS Effectiveness (comparison vs original)

In [ ]:
# Attempt to load original backtest metrics if available to compare
if os.path.exists(ORIG_BT_METRICS):
    orig = pd.read_json(ORIG_BT_METRICS)
    print('Original backtest metrics loaded')
    display(orig)
    # Build a simple comparison table
    comp = pd.DataFrame({'DBTS': metrics})
    try:
        comp['Original'] = orig.loc[['cumulative_return','annualized_return','annualized_vol','sharpe','sortino','max_drawdown','win_rate','n_trades']].T.values[0]
    except Exception:
        pass
    display(comp)
else:
    print('Original backtest metrics not found in outputs — cannot compare automatically')

## Section 11 — Conclusions (auto-generated)

In [ ]:
# Simple conclusions based on computed metrics
conclusions = []
if metrics['sharpe'] == metrics['sharpe'] and not math.isnan(metrics['sharpe']):
    conclusions.append(f'Strategy Sharpe (DBTS): {metrics[sharpe]:.2f}')
else:
    conclusions.append('Sharpe not available')
conclusions.append(f